# MNIST MLP3 — active-set Full Matrix-Log RG

This notebook uses the qualified MNIST/MLP3 SGD + Nesterov recipe and changes only the RG extension applied to `fc1.weight` and `fc2.weight`.

**Scientific hierarchy**

1. **Primary:** `mode="cone"`, `momentum_projection="projected_state"`, `normalization="self_consistent"`.
2. **Conservative control:** `mode="radial"` with projected momentum.
3. **Normalization ablation:** bulk-effective self-consistent `D_R` versus historical full `M`.
4. **Legacy-only ablations:** `mode="modewise"` and `momentum_projection="post_step"`; they are implemented but excluded from the primary grid.

The test set is never evaluated during hyperparameter selection. The final comparison uses the same three baseline seeds and run-level 95% Student-t intervals.


In [ ]:
from dataclasses import asdict, replace
from pathlib import Path
import json
import os
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

REPO = None
for path in [Path.cwd(), *Path.cwd().parents]:
    if (path / "baseline" / "rg_baselines").is_dir() and (
        path / "optimizers" / "full_matrix_log_rg" / "full_matrix_log_rg"
    ).is_dir():
        REPO = path.resolve()
        break
if REPO is None:
    raise RuntimeError("Run this notebook from a clone of CalculatedContent/rg_optimizers")

for path in [REPO / "baseline", REPO / "optimizers" / "full_matrix_log_rg"]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from rg_baselines import (
    BaselineConfig,
    DEFAULT_BASELINE_SEEDS,
    summarize_numeric_metrics,
)
from rg_baselines.engine import choose_device
from full_matrix_log_rg import FullMatrixLogConfig
from full_matrix_log_rg.experiment import run_mnist_sgd, run_validation_grid

DEVICE = choose_device()
DATA_DIR = Path(
    os.environ.get("RG_BASELINE_DATA_DIR", Path.home() / "rg-optimizer-data")
).expanduser().resolve()
RUN_ROOT = Path(
    os.environ.get(
        "RG_FML_RUN_ROOT",
        Path.home() / "rg-optimizer-runs" / "full_matrix_log_rg",
    )
).expanduser().resolve()
DATA_DIR.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print({"repo": str(REPO), "device": str(DEVICE), "data": str(DATA_DIR), "runs": str(RUN_ROOT)})


## Preflight

Run the geometry, active-set, normalization, projected-momentum, packaging, and restart tests before downloading MNIST.


In [ ]:
environment = dict(os.environ)
environment["PYTHONPATH"] = os.pathsep.join(
    [
        str(REPO / "optimizers" / "full_matrix_log_rg"),
        environment.get("PYTHONPATH", ""),
    ]
)
subprocess.run(
    [
        sys.executable,
        "-m",
        "unittest",
        "discover",
        "-s",
        str(REPO / "optimizers" / "full_matrix_log_rg" / "tests"),
        "-v",
    ],
    check=True,
    env=environment,
)


## Exact baseline recipe

Architecture, initialization, fixed 55k/5k split, Nesterov SGD, warm-up, cosine decay, clipping, WeightWatcher measurements, and checkpoint selection come from the qualified baseline package.


In [ ]:
BASE_CONFIG = BaselineConfig(
    optimizer="sgd_momentum",
    epochs=30,
    validation_size=5_000,
    sgd_learning_rate=0.05,
    sgd_min_learning_rate=5e-4,
    sgd_warmup_epochs=2,
    sgd_momentum=0.90,
    sgd_dampening=0.0,
    sgd_nesterov=True,
    sgd_weight_decay=1e-4,
    grad_clip_norm=1.0,
    ww_randomize=True,
    save_epoch_checkpoints=True,
)
SEEDS = tuple(DEFAULT_BASELINE_SEEDS)
TARGET_MATRICES = ("fc1.weight", "fc2.weight")
assert len(SEEDS) == 3 and len(set(SEEDS)) == 3
BASE_CONFIG.validate()
display(pd.DataFrame([asdict(BASE_CONFIG)]))


## Primary validation-only grid

The eight-point Mac grid directly tests the corrected scientific hierarchy:

- active-set cone versus radial potential control;
- self-consistent `D_R` versus full `M`;
- correction every 25 versus every 100 steps.

Projection strength is fixed at one, the correction cap at 10%, and momentum-state projection is always enabled. Set `RG_FML_SKIP_GRID=1` to reuse a completed `selected_config.json`.


In [ ]:
GRID_EPOCHS = int(os.environ.get("RG_FML_GRID_EPOCHS", "5"))
if GRID_EPOCHS < 3:
    raise ValueError("RG_FML_GRID_EPOCHS must be at least 3")
GRID_BASE = replace(
    BASE_CONFIG,
    seed=int(SEEDS[0]),
    epochs=GRID_EPOCHS,
    save_epoch_checkpoints=False,
)
GRID_CANDIDATES = [
    FullMatrixLogConfig(
        mode=mode,
        momentum_projection="projected_state",
        normalization=normalization,
        effective_rank_method="participation_ratio",
        normalization_gamma=0.0,
        projection_strength=1.0,
        max_correction_ratio=0.10,
        apply_every_steps=cadence,
        parameter_names=TARGET_MATRICES,
    )
    for mode in ("cone", "radial")
    for normalization in ("self_consistent", "full_m")
    for cadence in (25, 100)
]
GRID_ROOT = RUN_ROOT / "validation_grid"
selected_path = GRID_ROOT / "selected_config.json"

if os.environ.get("RG_FML_SKIP_GRID", "0") == "1":
    if not selected_path.is_file():
        raise FileNotFoundError(f"RG_FML_SKIP_GRID=1 but {selected_path} does not exist")
    payload = json.loads(selected_path.read_text(encoding="utf-8"))
    payload["parameter_names"] = (
        tuple(payload["parameter_names"]) if payload.get("parameter_names") else None
    )
    BEST = FullMatrixLogConfig(**payload)
    GRID = pd.read_csv(GRID_ROOT / "grid_results_ranked.csv")
else:
    grid_result = run_validation_grid(
        GRID_BASE,
        GRID_CANDIDATES,
        data_dir=DATA_DIR,
        output_dir=GRID_ROOT,
        device=DEVICE,
        progress=True,
        resume=True,
    )
    BEST = grid_result.selected_config
    GRID = grid_result.results

display(GRID)
display(
    GRID.groupby(["mode", "normalization"], as_index=False).agg(
        best_validation_loss=("validation_loss", "min"),
        best_validation_accuracy=("validation_accuracy", "max"),
        mean_abs_alpha_minus_2=("mean_abs_alpha_minus_2", "mean"),
    )
)
print("selected:", BEST)


## Final three-seed comparison

Each baseline and extended run is restartable. The extended optimizer projects the accepted Nesterov direction and rewrites the momentum buffer on every configured correction step.


In [ ]:
performance_frames = []
spectral_frames = []
correction_frames = []

for seed in SEEDS:
    config = replace(BASE_CONFIG, seed=int(seed))
    baseline = run_mnist_sgd(
        config,
        rg_config=None,
        data_dir=DATA_DIR,
        output_dir=RUN_ROOT / "final" / "sgd_momentum" / f"seed_{seed}",
        device=DEVICE,
        evaluate_test=True,
        progress=True,
        resume=True,
    )
    extended = run_mnist_sgd(
        config,
        rg_config=BEST,
        data_dir=DATA_DIR,
        output_dir=RUN_ROOT / "final" / "full_matrix_log_rg" / f"seed_{seed}",
        device=DEVICE,
        evaluate_test=True,
        progress=True,
        resume=True,
    )
    performance_frames.extend([baseline.performance, extended.performance])
    spectral_frames.extend([baseline.spectral, extended.spectral])
    if not extended.corrections.empty:
        correction_frames.append(extended.corrections)

PERFORMANCE = pd.concat(performance_frames, ignore_index=True)
SPECTRAL = pd.concat(spectral_frames, ignore_index=True, sort=False)
CORRECTIONS = (
    pd.concat(correction_frames, ignore_index=True, sort=False)
    if correction_frames
    else pd.DataFrame()
)

AGGREGATE = RUN_ROOT / "final" / "aggregate"
PLOT_DIR = AGGREGATE / "plots"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
PERFORMANCE.to_csv(AGGREGATE / "performance_by_epoch_and_seed.csv", index=False)
SPECTRAL.to_csv(AGGREGATE / "spectral_metrics_by_epoch_layer_and_seed.csv", index=False)
CORRECTIONS.to_csv(AGGREGATE / "rg_corrections_by_step.csv", index=False)


## Run-level 95% confidence intervals


In [ ]:
PERFORMANCE_SUMMARY = summarize_numeric_metrics(
    PERFORMANCE,
    group_columns=("run", "epoch"),
    metrics=(
        "train_loss",
        "validation_loss",
        "test_loss",
        "train_accuracy",
        "validation_accuracy",
        "test_accuracy",
    ),
)
SPECTRAL_OK = SPECTRAL[SPECTRAL["status"].eq("ok")].copy()
SPECTRAL_SUMMARY = summarize_numeric_metrics(
    SPECTRAL_OK,
    group_columns=("run", "layer", "epoch"),
    metrics=("alpha", "ERG_gap", "num_traps", "m_midpoint"),
)
PERFORMANCE_SUMMARY.to_csv(AGGREGATE / "performance_summary_95ci.csv", index=False)
SPECTRAL_SUMMARY.to_csv(AGGREGATE / "spectral_summary_95ci.csv", index=False)

required_performance_metrics = {
    "train_loss", "validation_loss", "test_loss",
    "train_accuracy", "validation_accuracy", "test_accuracy",
}
required_performance = PERFORMANCE_SUMMARY[
    PERFORMANCE_SUMMARY["metric"].isin(required_performance_metrics)
]
if set(required_performance["metric"]) != required_performance_metrics:
    raise RuntimeError("Missing required performance summary metrics")
if not required_performance["n"].eq(len(SEEDS)).all():
    raise RuntimeError("Performance confidence intervals do not contain all three runs")

required_spectral_metrics = {"alpha", "ERG_gap", "num_traps", "m_midpoint"}
required_spectral = SPECTRAL_SUMMARY[
    SPECTRAL_SUMMARY["metric"].isin(required_spectral_metrics)
]
if set(required_spectral["metric"]) != required_spectral_metrics:
    raise RuntimeError("Missing required direct WeightWatcher summary metrics")
if not required_spectral["n"].eq(len(SEEDS)).all():
    raise RuntimeError("Spectral confidence intervals do not contain all three runs")

display(PERFORMANCE_SUMMARY.sort_values(["metric", "run", "epoch"]))
display(SPECTRAL_SUMMARY.sort_values(["metric", "layer", "run", "epoch"]))


In [ ]:
for metric in ("validation_accuracy", "test_accuracy", "validation_loss", "test_loss"):
    summary = PERFORMANCE_SUMMARY[PERFORMANCE_SUMMARY["metric"].eq(metric)]
    figure, axis = plt.subplots(figsize=(9, 5))
    for run, group in summary.groupby("run"):
        group = group.sort_values("epoch")
        axis.plot(group["epoch"], group["mean"], linewidth=2.0, label=run)
        axis.fill_between(group["epoch"], group["ci_low"], group["ci_high"], alpha=0.16)
    axis.set(
        xlabel="Epoch",
        ylabel=metric.replace("_", " " ).title(),
        title=metric.replace("_", " " ).title(),
    )
    axis.grid(alpha=0.25)
    axis.legend(frameon=False)
    figure.tight_layout()
    figure.savefig(PLOT_DIR / f"{metric}_95ci.png", dpi=170, bbox_inches="tight")
    plt.show()

for metric in ("alpha", "ERG_gap", "num_traps"):
    summary = SPECTRAL_SUMMARY[SPECTRAL_SUMMARY["metric"].eq(metric)]
    for layer, layer_summary in summary.groupby("layer"):
        figure, axis = plt.subplots(figsize=(9, 5))
        for run, group in layer_summary.groupby("run"):
            group = group.sort_values("epoch")
            axis.plot(group["epoch"], group["mean"], linewidth=2.0, label=run)
            axis.fill_between(group["epoch"], group["ci_low"], group["ci_high"], alpha=0.16)
        if metric == "alpha":
            axis.axhline(2.0, linestyle="--", linewidth=1.0)
        axis.set(xlabel="Epoch", ylabel=metric, title=f"{layer}: {metric}")
        axis.grid(alpha=0.25)
        axis.legend(frameon=False)
        figure.tight_layout()
        figure.savefig(PLOT_DIR / f"{layer}_{metric}_95ci.png", dpi=170, bbox_inches="tight")
        plt.show()


## Cone, momentum, and normalization audit

A successful full-strength uncapped cone step should drive `max_signed_violation_after` close to zero. `momentum_buffer_correction_norm` verifies that the accepted flow was also written back into the optimizer state.


In [ ]:
if not CORRECTIONS.empty:
    CORRECTION_SUMMARY = CORRECTIONS.groupby(
        ["mode", "normalization", "parameter"], as_index=False
    ).agg(
        corrections=("correction_ratio", "size"),
        mean_correction_ratio=("correction_ratio", "mean"),
        max_correction_ratio=("correction_ratio", "max"),
        mean_base_potential_drift=("base_potential_drift", "mean"),
        mean_corrected_potential_drift=("corrected_potential_drift", "mean"),
        mean_max_violation_before=("max_signed_violation_before", "mean"),
        mean_max_violation_after=("max_signed_violation_after", "mean"),
        mean_active_set_size=("cone_active_set_size", "mean"),
        mean_active_set_iterations=("cone_iterations", "mean"),
        cone_converged_fraction=("cone_converged", "mean"),
        mean_momentum_buffer_correction=("momentum_buffer_correction_norm", "mean"),
        mean_normalization_dimension=("normalization_dimension", "mean"),
        mean_full_m_dimension=("full_m_dimension", "mean"),
        cap_fraction=("correction_capped", "mean"),
    )
    CORRECTION_SUMMARY.to_csv(AGGREGATE / "rg_correction_summary.csv", index=False)
    display(CORRECTION_SUMMARY)

required = [
    GRID_ROOT / "grid_results_ranked.csv",
    GRID_ROOT / "selected_config.json",
    AGGREGATE / "performance_by_epoch_and_seed.csv",
    AGGREGATE / "spectral_metrics_by_epoch_layer_and_seed.csv",
    AGGREGATE / "performance_summary_95ci.csv",
    AGGREGATE / "spectral_summary_95ci.csv",
]
for seed in SEEDS:
    for family in ("sgd_momentum", "full_matrix_log_rg"):
        run_dir = RUN_ROOT / "final" / family / f"seed_{seed}"
        required.extend(
            [
                run_dir / "checkpoint_latest.pt",
                run_dir / "checkpoint_best.pt",
                run_dir / "final_state.pt",
                run_dir / "run_complete.json",
            ]
        )
missing = [path for path in required if not path.is_file()]
if missing:
    raise RuntimeError("Missing required artifacts:\n" + "\n".join(map(str, missing)))
print("verified artifacts:", len(required))
